In [ ]:
# prepare_training_data.ipynb - Code Documentation
# ---------------------------------------------
# This script automates data preparation, fine-tuning, and evaluation of
# a Llama-based model for extracting medical entities from German
# cardiology letters. The following sections describe the code logic.

# 1. Environment Setup and Model Initialization
# ---------------------------------------------
# Import libraries and define device/precision settings.

from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template

import torch
import os
import json

import pandas as pd

from datasets import Dataset
from transformers import AutoTokenizer
from transformers import TrainingArguments, TextStreamer
from trl import SFTTrainer

from pydantic import BaseModel
from typing import Optional, List

In [ ]:
# Maximum length for tokenization
max_seq_length = 10226

In [ ]:
# Load the pre-trained Llama model and tokenizer using FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=None,
)
# Path or identifier for a refined/fine-tuned model (to be used later)
refined_model = "example_model"

# Data preparation

In [ ]:
import os
import random
from pathlib import Path
from shutil import copyfile

# Define the source folder containing original JSON files
source_folder = Path('data/json/annotations')

# Define the root and subfolders for train/test splits
split_root = Path('data/json/split')
train_folder = split_root / "train"
test_folder = split_root / "test"

# Create train and test folders if they do not exist
train_folder.mkdir(parents=True, exist_ok=True)
test_folder.mkdir(parents=True, exist_ok=True)

# List all JSON files in the source folder
json_files = [
    source_folder / f for f in os.listdir(source_folder)
    if f.endswith('.json') and (source_folder / f).is_file()
]

# Shuffle and split the files for train/test
random.seed(42)
random.shuffle(json_files) 
split_idx = int(0.8 * len(json_files)) 
train_files = json_files[:split_idx]
test_files = json_files[split_idx:]

# Copy files to their respective directories
for file in train_files:
    copyfile(file, train_folder / file.name)

for file in test_files:
    copyfile(file, test_folder / file.name)
    
# Print split summary
print(f"Split erfolgreich:")
print(f"Train-Dateien: {len(train_files)}")
print(f"Test-Dateien: {len(test_files)}")
print(f"Train-Pfad: {train_folder}")
print(f"Test-Pfad: {test_folder}")

In [ ]:
# Replace with the path to your folder
folder_path = 'data/json/split/train'

# Initialize list to store parsed data
train_dict_data = []

# Loop through JSON files and extract required fields
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    
    # Ensure it's a file and ends with .json
    if os.path.isfile(file_path) and filename.endswith('.json'):
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)

            # Access "text" and "annotations" keys
            text = data.get('text')
            annotations = data.get('annotations', [])

            # Remove 'begin' and 'end' keys from each annotation dict
            cleaned_annotations = [
                {k: v for k, v in annotation.items() if k not in ('begin', 'end')and v not in ('', None)}
                for annotation in annotations
            ]

            # Append to list in desired format
            train_dict_data.append({
                #'filename': filename,
                'prompt': text,
                'response': str(cleaned_annotations)
            })
            #print(f"Train-Dateien: {len(train_dict_data)}")


In [ ]:
# Replace with the path to your folder
folder_path = 'data/json/split/test'

# Initialize list to store parsed data
test_dict_data = []

# Loop through JSON files and extract required fields
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    
    # Ensure it's a file and ends with .json
    if os.path.isfile(file_path) and filename.endswith('.json'):
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)

            # Access "text" and "annotations" keys
            text = data.get('text')
            annotations = data.get('annotations', [])

            # Remove 'begin' and 'end' keys from each annotation dict
            cleaned_annotations = [
                {k: v for k, v in annotation.items() if k not in ('begin', 'end')and v not in ('', None)}
                for annotation in annotations
            ]

            # Append to list in desired format
            test_dict_data.append({
                #'filename': filename,
                'prompt': text,
                'response': str(cleaned_annotations)
            })

In [ ]:
# Optional: print or process the final list
for item in train_dict_data[:1]:
     print(item)

In [ ]:
# Optional: print or process the final list
for item in test_dict_data[:1]:
     print(item)

In [ ]:
# Create a Dataset from the dict
train_dataset = Dataset.from_pandas(pd.DataFrame(train_dict_data))
test_dataset = Dataset.from_pandas(pd.DataFrame(test_dict_data))

In [ ]:
class CardioScoreAnnotation(BaseModel):
    reasoning: Optional[str]
    entity_type: Optional[str]
    typ: Optional[str]
    aktualitaet: Optional[str]
    numeral: Optional[float]
    unit: Optional[str]
    #scores: Optional[str]
    annotated_span: str

# Document model
class CardioScoreDocument(BaseModel):
    text: str
    annotations: List[CardioScoreAnnotation]

In [ ]:
schema = json.dumps(CardioScoreDocument.model_json_schema())

# Define system message
system_message = f"""You are a physician. Your task is to extract medication information with entity_types:

Alter, Antithrombotic treatment, Bempedoinsäure, Blutdruck (systolisch) (RR), Cholesterin, Diabetes,  Mellitus, Ezetimib, Gefäßerkrankung, Geschlecht, HDL Cholesterin, HbA1c, Herzinsuffizienz, Hypertonie, LDL Cholesterin, Nikotinkonsum, SGLT2 Inhibitoren, Schlaganfall/TIA, Scores, Scores (explicit), Statinse, GFR CKD-EPI

from a given German doctoral letter from cardiology. In addition extract the related information for reasoning, typ, aktualitaet, numeral, unit, scores IF applicable.

If there is NO medication information for an entity_type IGNORE it.
Ignore the <[Pseudo] > tags aroung dates, e.g. <[Pseudo] 2000-04-09 > is 2000-04-09.
ONLY respond with an instance of JSON without any additional information. You have access to a JSON schema, which will determine how the JSON should be structured.
Make sure to return ONLY an instance of the JSON, NOT the schema itself. Do not add any additional information.
JSON schema:
{schema}
"""
print(schema)

In [ ]:
# Convert into chat template
def create_conversation(sample):
    # Create conversation with system, user, and assistant
    conversation = {
        "messages": [
            {"role": "system", "content": system_message},  # System message
            {"role": "user", "content": sample["prompt"]},  # User prompt
            {"role": "assistant", "content": sample["response"]}  # Assistant's expected response
        ]
    }

    return conversation

# Apply the function to the dataset
train_chat_dataset = train_dataset.map(create_conversation, remove_columns=['prompt', 'response'])
test_chat_dataset = test_dataset.map(create_conversation, remove_columns=['prompt', 'response'])

In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
)

def apply_template(examples):
    messages = examples["messages"]
    text = [tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=False) for message in messages]
    return {"text": text}

final_train = train_chat_dataset.map(apply_template, batched=True)
final_test = test_chat_dataset.map(apply_template, batched=True)

# Model Training

In [ ]:
print("Training....")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth"
)

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=final_train,  # 80% of the data
    eval_dataset=final_test,    # 20% of the data
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    data_collator=data_collator, 
    args=TrainingArguments(
        learning_rate=3e-4,
        lr_scheduler_type="linear",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=100,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        output_dir=refined_model,
        seed=0,
        save_steps=100,
        eval_steps=250,
        save_total_limit=2,
        save_strategy="steps",                 
        #evaluation_strategy="steps",
        #load_best_model_at_end=True,
        #metric_for_best_model="loss",
        #greater_is_better=False,
    ),
)


In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained("models/training/final_model")


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained("models/training/final_model", max_seq_length = 10226)